This is the beginning of our bronze data ingestion into DuckDB. Our first cell assigns our project root, raw data directory, database path, and the raw data file.

In [ ]:
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "ltv.duckdb"

RAW_DIR.mkdir(parents=True, exist_ok=True)  

raw_file = RAW_DIR / "online_retail_raw_data.xlsx"
print(raw_file.exists())

True


This cell imports DuckDB and pandas, and then concats the different sheets from our raw data into one table to use for our silver and gold notebooks. We also create our bronze table in our database.

In [ ]:
##run pkill -f ipykernel if you are getting a locking error

import duckdb
import pandas as pd

#Read every sheet (2009-2010 and 2010-2011) into a dict of DataFrames
sheets = pd.read_excel(raw_file, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
for name, df in sheets.items():
    print(name, df.shape)

#Stack the sheets into one table (values unchanged)
bronze_df = pd.concat(sheets.values(), ignore_index=True)

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE OR REPLACE TABLE bronze AS SELECT * FROM bronze_df")
print(con.sql("SELECT COUNT(*) AS rows FROM bronze").fetchone())
con.close()

Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)
(1067371,)


This cell is a spot check to ensure our bronze table has been created as expected.

In [3]:
with duckdb.connect(str(DB_PATH)) as con:
    display(con.sql("SHOW TABLES").df())

,name
0,bronze
